# Project 2: House Price Prediction
**Задача:** предсказать цену дома по нескольким признакам (multiple regression + scaling + Pipeline).
**Темы:** multiple regression, StandardScaler, Pipeline, метрики, важность признаков.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

rng = np.random.default_rng(7)
n = 300
df = pd.DataFrame({
    "площадь":       rng.uniform(40, 200, n).round(0),
    "комнаты":       rng.integers(1, 6, n),
    "возраст_дома":  rng.integers(0, 50, n),
    "до_центра_км":  rng.uniform(0.5, 25, n).round(1),
})
df["цена"] = (1.8*df["площадь"] + 12*df["комнаты"] - 0.9*df["возраст_дома"]
              - 3.5*df["до_центра_км"] + 120 + rng.normal(0, 18, n)).round(1)
df.head()

In [ ]:
# EDA
print(df.describe().round(1))
# Корреляции: какой признак сильнее связан с ценой
print("\nКорреляция с ценой:")
print(df.corr()["цена"].drop("цена").round(2).sort_values())
df.hist(figsize=(10, 6), bins=20); plt.tight_layout(); plt.show()

In [ ]:
X = df.drop(columns="цена")
y = df["цена"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Pipeline: scaler + модель одним объектом. Защита от data leakage встроена:
# fit масштабирует ТОЛЬКО train, predict сам применяет transform
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression()),
])
pipe.fit(X_tr, y_tr)
y_pred = pipe.predict(X_te)
print(f"MAE = {mean_absolute_error(y_te, y_pred):.1f} тыс.$ | R2 = {r2_score(y_te, y_pred):.3f}")

# Веса на МАСШТАБИРОВАННЫХ признаках можно честно сравнивать между собой
w = pd.Series(pipe.named_steps["model"].coef_, index=X.columns).sort_values()
plt.barh(w.index, w.values); plt.title("Влияние признаков (масштабированные веса)")
plt.grid(True, axis='x'); plt.show()

In [ ]:
# Error analysis + предсказание для новой квартиры
res = y_te - y_pred
plt.scatter(y_pred, res, alpha=0.6); plt.axhline(0, c='r', ls='--')
plt.xlabel('предсказание'); plt.ylabel('остаток'); plt.title('Остатки без узоров = ок'); plt.grid(True); plt.show()

new = pd.DataFrame([[85, 3, 10, 5.0]], columns=X.columns)
print("Новая квартира 85м2/3к/10лет/5км -> цена:", pipe.predict(new).round(1)[0], "тыс.$")

## Выводы
- Pipeline = масштабирование + модель в одном объекте, безопасно и удобно.
- После scaling веса сравнимы: площадь — главный фактор «+», расстояние и возраст — «−».
- Остатки без узоров — модель не врёт систематически.

## Задания
1. Добавь Ridge(alpha=10) вместо LinearRegression в Pipeline — изменились ли метрики?
2. Добавь «выбросы» (3 дома с ценой x3) и посмотри, как испортится MAE.
3. Добавь категориальный признак «район» через pd.get_dummies.